In [82]:
import sys
import os

from langchain.tools import tool
from langchain.tools import Tool
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from tools.simple_openai_tool import add_numbers

sys.path.append(os.path.abspath(".."))

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

@tool
def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string, negates the first number, and successively subtracts 
    the remaining numbers in the list.

    This function is designed to handle input in string format, where numbers are separated 
    by spaces, commas, or other delimiters. It parses the string, extracts valid numeric values, 
    and performs a step-by-step subtraction operation starting with the first number negated.

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string may include spaces, commas, or 
      other delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": -130}

    Notes:
    - Non-numeric characters in the input are ignored.
    - If the input string contains only one valid number, the result will be that number negated.
    - Handles a variety of delimiters (e.g., spaces, commas) but does not validate input formats 
      beyond extracting numeric values.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number negated
    result = -1 * numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}




@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

@tool
def divide_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates the result of dividing the first number 
    by the subsequent numbers in sequence.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the quotient.

    Example Input:
    "100, 5, 2"

    Example Output:
    {"result": 10.0}

    Notes:
    - If no numbers are found, the result defaults to 0.
    - Division by zero will raise an error.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]


    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Calculate the result of dividing the first number by subsequent numbers
    result = numbers[0]
    for num in numbers[1:]:
        result /= num

    return {"result": result}


@tool
def new_subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

    This function is designed to handle input in string format, where numbers may be separated by spaces, 
    commas, or other delimiters. It parses the input string, extracts numeric values, and calculates 
    the result by subtracting each subsequent number from the first. inputs[0]-inputs[1]-inputs[2]

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string can include spaces, commas, or other 
      delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Usage:
    - Input: "100, 20, 10"
    - Output: {"result": 70}

    Limitations:
    - The function does not handle cases where numbers are formatted with decimals or other non-integer representations.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number
    result = numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}


In [83]:
tools = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers]
tools

[StructuredTool(name='add_numbers', description='Adds a list of numbers provided in the input dictionary or extracts numbers from a string.\n\nParameters:\n- inputs (str): \nstring, it should contain numbers that can be extracted and summed.\n\nReturns:\n- dict: A dictionary with a single key "result" containing the sum of the numbers.\n\nExample Input (Dictionary):\n{"numbers": [10, 20, 30]}\n\nExample Input (String):\n"Add the numbers 10, 20, and 30."\n\nExample Output:\n{"result": 60}', args_schema=<class 'langchain_core.utils.pydantic.add_numbers'>, func=<function add_numbers at 0x10ba1c820>),
 StructuredTool(name='new_subtract_numbers', description='Extracts numbers from a string and performs subtraction sequentially, starting with the first number.\n\nThis function is designed to handle input in string format, where numbers may be separated by spaces, \ncommas, or other delimiters. It parses the input string, extracts numeric values, and calculates \nthe result by subtracting eac

In [84]:
from langgraph.prebuilt import create_react_agent
math_agent = create_react_agent(
    model= llm,
    tools=tools,
    prompt="You are a helpful mathematical assistant that can perform various operations. " \
    "Use the tools precisely and explain your reasoning clearly."
)

In [85]:


response= math_agent.invoke({
    "messages": [("human", "what is 24 divide by 4 ?")]
})

#Get the final answer
final_answer= response["messages"][-1].content
print(final_answer)

The result of dividing 24 by 4 is 6.0.


In [86]:
response_2 = math_agent.invoke({
    "messages": [("human", "Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-1].content
print(final_answer_2)

The result of subtracting 20 and 10 from 100 is 70.


In [87]:
print("\n--- Testing MultiplyTool ---")
response = math_agent.invoke({
    "messages": [("human", "Multiply 2, 3, and four.")]
})
print("Agent Response:", response["messages"][-1].content)

print("\n--- Testing DivideTool ---")
response = math_agent.invoke({
    "messages": [("human", "Divide 100 by 5 and then by 2.")]
})
print("Agent Response:", response["messages"][-1].content)


--- Testing MultiplyTool ---
[2, 3, 4]
2
3
4
Agent Response: The product of 2, 3, and 4 is 24.

--- Testing DivideTool ---
Agent Response: The result of dividing 100 by 5 and then by 2 is 10.0.


In [88]:
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers]

In [89]:

# Create the agent with all tools
math_agent_new = create_react_agent(
    model=llm,
    tools=tools_updated,
    # Optional: Add a system message to guide the agent's behavior
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)
print("agent",math_agent_new)

agent <langgraph.graph.state.CompiledStateGraph object at 0x1134f9a60>


In [81]:
# Test Cases
test_cases = [
    {
        "query": "Subtract 100, 20, and 10.",
        "expected": {"result": 70},
        "description": "Testing subtraction tool with sequential subtraction."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }

]

In [90]:
correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = math_agent_new.invoke({"messages": [("human", query)]})
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['add_numbers', 'new_subtract_numbers', 'multiply_numbers', 'divide_numbers']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content)["result"]
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)


--- Test Case 1: Testing subtraction tool with sequential subtraction. ---
Query: Subtract 100, 20, and 10.
Tool Result: 70
Expected Result: 70
✅ Test Passed: Testing subtraction tool with sequential subtraction.

--- Test Case 2: Testing multiplication tool for a list of numbers. ---
Query: Multiply 2, 3, and 4.
[2, 3, 4]
2
3
4
Tool Result: 24
Expected Result: 24
✅ Test Passed: Testing multiplication tool for a list of numbers.

--- Test Case 3: Testing division tool with sequential division. ---
Query: Divide 100 by 5 and then by 2.
Tool Result: 10.0
Expected Result: 10.0
✅ Test Passed: Testing division tool with sequential division.

--- Test Case 4: Testing subtraction tool with negative results. ---
Query: Subtract 50 from 20.
Tool Result: -30
Expected Result: -30
✅ Test Passed: Testing subtraction tool with negative results.

Correctly passed tests: ['Testing subtraction tool with sequential subtraction.', 'Testing multiplication tool for a list of numbers.', 'Testing division t